In [1]:
import os
import pandas as pd
import gzip
import COG_enrichment as ce
import kegg_enrichment as ke
import pickle
from scipy.stats import fisher_exact


#MGYG000000003_1	Prodigal:2.6	CDS	23749	24648	.	-	0	ID=MGYG000000003_00025;Name=fieF;db_xref=COG:COG0053;gene=fieF;inference=ab initio prediction:Prodigal:2.6,similar to AA sequence:UniProtKB:P69380;locus_tag=MGYG000000003_00025;product=Ferrous-iron efflux pump FieF;eggNOG=679935.Alfi_1785;COG=P;KEGG=-;Pfam=PF01545;InterPro=IPR027469,IPR002524
#MGYG000000003_1	Prodigal:2.6	CDS	24855	26339	.	-	0	ID=MGYG000000003_00026;eC_number=6.1.1.15;Name=proS;db_xref=COG:COG0442;gene=proS;inference=ab initio prediction:Prodigal:2.6,similar to AA sequence:UniProtKB:Q5SM28;locus_tag=MGYG000000003_00026;product=Proline--tRNA ligase;eggNOG=679935.Alfi_1784;COG=J;KEGG=ko:K01881;Pfam=PF00587,PF03129,PF09180;InterPro=IPR016061,IPR017449,IPR004154,IPR002314,IPR033721,IPR036621,IPR006195,IPR004499

infile = '../../crc_analysis_12177/data/merged_hgt.csv'
db_idir = '../../HGT_demo_file/DB.genome_annotation'
infile2 = '../../crc_analysis_12177/data/metadata.tsv'
pfile = '../../HGT_demo_file/ko_pathway_dict.pickle'
groupid = 'phenotype'
outdir = '.'
fr_size = 1000
with open(pfile, 'rb') as f: 
    ko_pathway_dict = pickle.load(f)
df = pd.read_csv(infile, header=0, index_col=None)
df.rename(columns={'receptor':'recipient'}, inplace=True)
metadata = pd.read_csv(infile2, header=0, index_col=0, sep='\t')
if len(metadata[groupid].unique()) != 2:
    print('Error: the column {} does not have exact 2 level.'.format(groupid))
    exit(1)
hgt_slist = list(set(df['sample']))
g_slist = list(metadata.index)
valid = True
for s in list(hgt_slist):
    if s not in g_slist:
        print('Error: group information of sample {} in HGT event dose NOT exist.'.format(s))
        valid = False
if not valid:
    exit(2)

In [12]:
def gz2df(ifile):
    with gzip.open(ifile, 'rb') as f:
        df = pd.read_csv(f, header=None, index_col=None, sep='\t')
    return df

def extract_line(content):
    ko_list = []
    cog_list = []
    cog_cate_list = []
    # check kegg
    if 'KEGG=' in content:
        kegg = content.split('KEGG=')[1].strip().split(';')[0]
        kos = kegg.split(',')
        #print('kos', kos)
        for ko in kos:
            if ko.startswith('ko:'):
                ko = ko.split('ko:')[-1]
                ko_list.append(ko)
    
    if 'db_xref=' in content:
        cog = content.split('db_xref=')[1].strip().split(';')[0]
        cogs = cog.split(',')
        for cog in cogs:
            if cog.startswith('COG:'):
                cog = cog.split('COG:')[-1]
                cog_list.append(cog)

    if 'COG=' in content:
        cog_cate = content.split('COG=')[1].strip().split(';')[0]
        cog_cate = cog_cate.split(',')[0]
        for cc in cog_cate:
            if cc != '-':
                cog_cate_list += [char for char in cc]
    return ko_list, cog_list, cog_cate_list

def extract(df):
    ko_list = []
    cog_list = []
    cog_cate_list = []
    for idx in df.index:
        kos, cogs, cog_cates = extract_line(df.loc[idx, 8])
        ko_list += kos
        cog_list += cogs
        cog_cate_list += cog_cates
    return ko_list, cog_list, cog_cate_list

def overlap(range1, range2):
    if range1[0] > range2[1] or range1[1] < range2[0]:
        return False
    else:
        return True

# styp = recipient or donor
def search_event(scaffold, db, range):
    scaffold_df = db[db[0] == scaffold]
    valid_idx = []
    for idx in scaffold_df.index:
        if overlap(range, [scaffold_df.loc[idx, 3], scaffold_df.loc[idx, 4]]):
            valid_idx.append(idx)
    valid_df = scaffold_df.loc[valid_idx, ]
    return valid_df
    
def search_row(idx, df, db_dir, fr_size):
    tmp = '{}.gff.gz'
    row = df.loc[idx, ]
    # for recipient
    recipient = row['recipient']
    chrom = recipient.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    
    if not os.path.exists(ifile):
        recipient_df = pd.DataFrame()
        recipient_excluded_df = pd.DataFrame()
    else:
        db = gz2df(ifile)
        recipient_range = [max(0, row['insert_locus']-fr_size), row['insert_locus']+fr_size]
        recipient_df = search_event(recipient, db, recipient_range)
        recipient_excluded_df = get_backgroud(recipient, db, recipient_range)
    # for donor
    donor = row['donor']
    range = [max(0, row['delete_start'] - fr_size), row['delete_end'] + fr_size]
    chrom = donor.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    if not os.path.exists(ifile):
        donor_df = pd.DataFrame()
        donor_excluded_df = pd.DataFrame()
    else:
        db = gz2df(ifile)
        donor_df = search_event(donor, db, range)
        donor_excluded_df = get_backgroud(donor, db, range)
    return recipient_df, donor_df, recipient_excluded_df, donor_excluded_df 

def get_backgroud(scaffold, db, range_excluded):
    scaffold_df = db[db[0] == scaffold]
    valid_idx = []
    for idx in scaffold_df.index:
        if not overlap(range_excluded, [scaffold_df.loc[idx, 3], scaffold_df.loc[idx, 4]]):
            valid_idx.append(idx)
    valid_df = scaffold_df.loc[valid_idx, ]
    return valid_df

def enrichment(metadata, result_anno, groupid, type='KEGG'):
    pheno_set = list(set(metadata[groupid]))
    g1 = pheno_set[0]
    g2 = pheno_set[1]
    vf_df = result_anno[(result_anno['recipient_{}_n'.format(type)]>0) | (result_anno['donor_{}_n'.format(type)]>0)]
    cate_df = pd.DataFrame(columns=[g1, g2])
    for idx in vf_df.index:
        recipient_VF_category = vf_df.loc[idx, 'recipient_{}_category'.format(type)].split(';')
        donor_VF_category = vf_df.loc[idx, 'donor_{}_category'.format(type)].split(';')
        all_cates = recipient_VF_category+donor_VF_category
        for cate in all_cates:
            if cate == 'NA':
                continue
            if cate not in cate_df.index:
                cate_df.loc[cate, g1] = 0
                cate_df.loc[cate, g2] = 0
            cate_df.loc[cate, metadata.loc[vf_df.loc[idx, 'sample'], groupid]] += 1
    pvalue_reformat = pd.DataFrame(columns=['group1', 'group2', 'category', 'g1_in_category', 'g1_total', 'g2_in_category', 'g2_total', 'pvalue', 'odds_ratio'])
    g1_total = cate_df[g1].sum()
    g2_total = cate_df[g2].sum()
    for cate in cate_df.index:
        a = cate_df.loc[cate, g1]
        b = cate_df.loc[cate, g2]
        c = g1_total - a
        d = g2_total - b
        oddsratio, pvalue = fisher_exact([[a, b], [c, d]])
        pvalue_reformat.loc[cate, ] = [g1, g2, cate, a, g1_total, b, g2_total, pvalue, oddsratio]
    pvalue_reformat.fillna('NA', inplace=True)
    return pvalue_reformat

In [8]:
result_anno = pd.DataFrame(columns=['id', 'sample', 'recipient_KEGG_n', 'recipient_KEGG_list', 'recipient_COG_n', 'recipient_COG_list',
                                    'donor_KEGG_n', 'donor_KEGG_list', 'donor_COG_n', 'donor_COG_list',
                                    'recipient', 'insert_locus', 'donor', 'delete_start', 'delete_end', 'reverse_flag'])

bk_ko_set = set()
bk_cog_set = set()
exist_ko_set = set()
exist_cog_set = set()

for idx in df.index:
    recipient_df, donor_df, recipient_excluded_df, donor_excluded_df = search_row(idx, df, db_idir, fr_size)
    id = 'HGT_c{}'.format(idx+1)
    sample = df.loc[idx, 'sample']
    ko_list, cog_list, cog_cate = extract(recipient_df)
    exist_ko_set.update(set(ko_list))
    exist_cog_set.update(set(cog_cate))
    recipient_KEGG_n = len(set(ko_list))
    recipient_COG_n = len(set(cog_list))
    recipient_KEGG_list = ';'.join(set(ko_list))
    recipient_COG_list = ';'.join(set(cog_list))
    if recipient_KEGG_n == 0:
        recipient_KEGG_list = 'NA'
    if recipient_COG_n == 0:
        recipient_COG_list = 'NA'

    ko_list, cog_list, cog_cate = extract(donor_df)
    exist_ko_set.update(set(ko_list))
    exist_cog_set.update(set(cog_cate))
    donor_KEGG_n = len(set(ko_list))
    donor_COG_n = len(set(cog_list))
    donor_KEGG_list = ';'.join(set(ko_list))
    donor_COG_list = ';'.join(set(cog_list))
    if donor_KEGG_n == 0:
        donor_KEGG_list = 'NA'
    if donor_COG_n == 0:
        donor_COG_list = 'NA'
    recipient = df.loc[idx, 'recipient']
    insert_locus = df.loc[idx, 'insert_locus']
    donor = df.loc[idx, 'donor']
    delete_start = df.loc[idx, 'delete_start']
    delete_end = df.loc[idx, 'delete_end']
    reverse_flag = df.loc[idx, 'reverse_flag']
    result_anno.loc[len(result_anno), ] = [id, sample, recipient_KEGG_n, recipient_KEGG_list, recipient_COG_n, recipient_COG_list, donor_KEGG_n, donor_KEGG_list, donor_COG_n, donor_COG_list, recipient, insert_locus, donor, delete_start, delete_end, reverse_flag]

    # add bg ko
    ko_list, cog_list, cog_cate = extract(recipient_excluded_df)
    bk_ko_set.update(set(ko_list))
    bk_cog_set.update(set(cog_cate))

    ko_list, cog_list, cog_cate = extract(donor_excluded_df)
    bk_ko_set.update(set(ko_list))
    bk_cog_set.update(set(cog_cate))
    
result_anno.to_csv(os.path.join(outdir, 'output.functional_annotation.annotated.tsv'), index=False, sep='\t')


In [14]:
enrichment(metadata, result_anno, groupid)


KeyError: 'recipient_KEGG_category'

In [19]:
ke.get_pathways(result_anno.loc[279, 'recipient_KEGG_list'].split(';'), ko_pathway_dict)

defaultdict(int, {'map03010': 1, 'ko03010': 1})

In [20]:
result_anno

,id,sample,recipient_KEGG_n,recipient_KEGG_list,recipient_COG_n,recipient_COG_list,donor_KEGG_n,donor_KEGG_list,donor_COG_n,donor_COG_list,recipient,insert_locus,donor,delete_start,delete_end,reverse_flag
0,HGT_c1,SAMEA3541470,0,NA,0,NA,0,NA,0,NA,MGYG000001378_5,162024,MGYG000001977_266,401,3131,False
1,HGT_c2,SAMEA3541470,0,NA,0,NA,0,NA,0,NA,MGYG000000041_32,23461,MGYG000002285_26,16304,17077,True
2,HGT_c3,SAMEA3541470,0,NA,1,COG2315,0,NA,0,NA,MGYG000001346_15,1222520,MGYG000003613_79,237,2609,False
3,HGT_c4,SAMEA3541470,0,NA,0,NA,0,NA,0,NA,MGYG000004271_5,51875,MGYG000003892_37,27061,28809,True
4,HGT_c5,SAMEA3541470,0,NA,0,NA,0,NA,0,NA,MGYG000002560_3,86098,MGYG000004055_26,6438,9869,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,HGT_c280,SAMEA3541592,2,K02357;K02967,1,COG0264,0,NA,0,NA,MGYG000000003_2,439630,MGYG000003084_128,576,1931,True
280,HGT_c281,SAMEA3541593,0,NA,0,NA,0,NA,0,NA,MGYG000002393_14,66641,MGYG000004558_8,120008,121918,False
281,HGT_c282,SAMEA3541593,0,NA,0,NA,2,K13963;K07496,0,NA,MGYG000004785_168,3425,MGYG000001338_1,2158124,2160152,False
282,HGT_c283,SAMEA3541593,0,NA,0,NA,0,NA,0,NA,MGYG000000941_2,72820,MGYG000000851_5,13488,15072,False
